# Customize Tuner for Neural Network Optimization 
In this notebook, we will look at how to customize hyperparameter optimization for a model:
- Use different hyperparameter selection, or settings for the standard models
- How to define a custom model, and use it in the hyperparameter optimization
- How to use the pipeline-utility classes `PipelineWithHyperparameterRooting` and `ColumnTransformerWithHyperparameterRooting` 

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from torch import nn
from six import iteritems

import mother.optimization as opt

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2)
print(
    f"Train / Validation samples = {X_train.shape[0]} / {X_valid.shape[0]} with {X_train.shape[1]} features and {y_train.nunique()} labels."
)

In [ ]:
from collections import OrderedDict
from mother.ml import AbstractMotherPipeline
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
import torch
from mother.ml.models.utils import add_prefix_to_dict_keys
from optuna.trial import Trial
import pandas as pd
from sklearn.metrics import accuracy_score


class TorchNNMother(AbstractMotherPipeline):
    def __init__(self, lr: float = 1e-3, **kwargs) -> None:

        self.network = nn.Sequential(
            OrderedDict(
                [
                    ("linear1", nn.Linear(in_features=30, out_features=10)),
                    ("relu", nn.ReLU()),
                    ("linear2", nn.Linear(in_features=10, out_features=1)),
                    ("sigmoid", nn.Sigmoid()),
                ]
            )
        )

        self.optimizer = Adam(self.network.parameters(), lr=lr)
        self.lr = lr
        self.loss = nn.BCELoss()
        self._init_params: dict = {"lr": lr}

        non_optimised_params: list[str] = ["_init_params"]
        for k, v in kwargs.items():
            if k not in non_optimised_params:
                self._init_params[k] = v

    def default_parameters(self, prefix: str = "") -> dict:
        return add_prefix_to_dict_keys({"lr": 1e-3}, prefix=prefix)

    def get_hyperparameter_space(self, X, y, trial: Trial, prefix: str = "") -> dict:
        suggested_params: dict = {"lr": trial.suggest_float(prefix + "lr", 1e-5, 1e-3, log=True)}
        suggested_params = add_prefix_to_dict_keys(suggested_params, prefix=prefix)

        return suggested_params

    def get_params(self, deep=True) -> dict:
        return self._init_params

    def set_params(self, **params):
        for key, value in iteritems(params):
            if key in self._init_params.keys():
                self._init_params[key] = value

        # Keep runtime attributes and optimizer in sync with tuned params.
        self.lr = float(self._init_params["lr"])

        return self.__init__(**self._init_params)

    def _get_dataloader(self, X, y, batch_size: int = 128, shuffle: bool = False) -> DataLoader:
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        return DataLoader(
            TensorDataset(
                torch.tensor(X).to(torch.float32),
                torch.tensor(y).reshape(-1, 1).to(torch.float32),
            ),
            batch_size=batch_size,
            shuffle=shuffle,
        )

    def validation(self, X_valid, y_valid):
        valid_data_loader = self._get_dataloader(X_valid, y_valid)

        self.network.eval()
        valid_loss = 0.0
        valid_acc = 0.0

        with torch.no_grad():
            for X, y in valid_data_loader:
                y_hat = self.network(X)
                valid_loss += self.loss(y_hat, y).item() / len(valid_data_loader)
                y_hat_pred = (y_hat > 0.5).float()
                valid_acc += accuracy_score(
                    y.detach().numpy().ravel(),
                    y_hat_pred.detach().numpy().ravel(),
                ) / len(valid_data_loader)

        return valid_loss, valid_acc

    def fit(self, X_train, y_train, n_epochs: int = 100):
        train_data_loader = self._get_dataloader(X_train, y_train, shuffle=True)

        # Recreate optimizer so each fit starts from current tuned lr and clean state.
        self.optimizer = Adam(self.network.parameters(), lr=float(self._init_params["lr"]))

        for epoch in range(n_epochs):
            self.network.train()
            train_loss = 0.0
            for X, y in train_data_loader:
                self.optimizer.zero_grad()
                y_hat = self.network(X)
                batch_train_loss = self.loss(y_hat, y)

                if not torch.isfinite(batch_train_loss):
                    raise ValueError("Training diverged (non-finite loss). Try a lower learning rate.")

                batch_train_loss.backward()
                self.optimizer.step()
                train_loss += batch_train_loss.item() / len(train_data_loader)
        return self


model = TorchNNMother(lr=1e-3)
model.fit(X_train, y_train)
print(model.validation(X_valid, y_valid))

(0.2540920674800873, 0.9035087719298246)


In [ ]:
from optuna import Trial
import sklearn.base as skl_base
import mother.optimization as opt


class TorchMotherTuner(opt.AbstractMotherTuner):
    def __init__(self, **kwargs):
        # create a customised scorer
        super().__init__(**kwargs)

    def objective(self, trial: Trial, context: opt.ObjectiveContext) -> float:
        X_train, X_valid, y_train, y_valid = train_test_split(context.X, context.y, test_size=0.2)
        print(
            f"Train / Validation samples = {X_train.shape[0]} / {X_valid.shape[0]} with {X_train.shape[1]} features and {y_train.nunique()} labels."
        )

        # fit
        estimator = skl_base.clone(context.estimator)
        suggested_params_to_train: dict = context.get_hyper_space(trial=trial, X=context.X, y=context.y)
        estimator.set_params(**suggested_params_to_train)
        estimator.fit(X_train, y_train)

        # calculate valid loss
        valid_loss, valid_acc = context.estimator.validation(X_valid, y_valid)

        return valid_loss

    def call_optimize(self, context: opt.ObjectiveContext) -> None:
        self.study.optimize(
            lambda trial: self.objective(trial, context=context),
            n_trials=self.n_trials_optuna,
            gc_after_trial=True,
            callbacks=self.get_callbacks(),
        )


tuner = TorchMotherTuner(
    n_trials_optuna=3,  # number of trials for hyperparameter optimization
    n_threads_optuna=10,  # parallel threads for cross-validation evaluation
    n_startup_trials=1,  # number of random trials before using optuna
    tuning_direction="minimize",  # Need to minimize the loss!
)

model_tuned = tuner.optimize(
    model,
    X,
    y,
    cross_validation=None,
    hyperparameter_space_function=model.get_hyperparameter_space,
)

/workspaces/MotherML/src/mother/optimization/core.py:131: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:131: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:131: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(


Train / Validation samples = 455 / 114 with 30 features and 2 labels.
{'lr': 0.001}
Train / Validation samples = 455 / 114 with 30 features and 2 labels.
{'lr': 0.0009361658778024464}
Train / Validation samples = 455 / 114 with 30 features and 2 labels.
{'lr': 0.0008048266581936503}
FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.22355324029922485], datetime_start=datetime.datetime(2026, 7, 17, 14, 11, 33, 456914), datetime_complete=datetime.datetime(2026, 7, 17, 14, 11, 34, 358672), params={'lr': 0.001}, user_attrs={}, system_attrs={'fixed_params': {'lr': 0.001}}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.001, log=True, low=1e-05, step=None)}, trial_id=0, value=None)
{'lr': 0.001}
{'lr': 0.001}
